In [69]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from pickle import load

In [70]:
df = pd.read_csv(os.path.join('..','data','df_final_for_metadata_prediction.csv'))
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200452 entries, 0 to 200451
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0    200452 non-null  int64  
 1   path          200452 non-null  object 
 2   subject_id    200452 non-null  int64  
 3   gender        200452 non-null  object 
 4   insurance     200452 non-null  object 
 5   grouped_race  200452 non-null  object 
 6   age_decile    200452 non-null  object 
 7   No Finding    200452 non-null  float64
dtypes: float64(1), int64(2), object(5)
memory usage: 12.2+ MB


In [71]:
patients_df = pd.read_csv(os.path.join('..','data','patients.csv'))
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 382278 entries, 0 to 382277
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   subject_id         382278 non-null  int64 
 1   gender             382278 non-null  object
 2   anchor_age         382278 non-null  int64 
 3   anchor_year        382278 non-null  int64 
 4   anchor_year_group  382278 non-null  object
 5   dod                9509 non-null    object
dtypes: int64(3), object(3)
memory usage: 17.5+ MB


In [72]:
extracted_embedding_folder = os.path.join("..", "data", "generalized-image-embedding-all")
train_test_idx_path = os.path.join(extracted_embedding_folder, "train_test_idx.pkl")

if os.path.exists(train_test_idx_path):
    with open(train_test_idx_path, 'rb') as f:
        train_indices, test_indices, train_all_ids, test_all_ids = load(f)
    print("train and test indices loaded from pickle file.")
    train_ids = np.unique(train_all_ids)
    test_ids = np.unique(test_all_ids)


train and test indices loaded from pickle file.


In [73]:
# Merging A and B with left join, keeping all 'subject_id' from A and only 'anchor_age' from B
df = pd.merge(df, patients_df[['subject_id', 'anchor_age']], on='subject_id', how='left')

# Display the merged dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200452 entries, 0 to 200451
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0    200452 non-null  int64  
 1   path          200452 non-null  object 
 2   subject_id    200452 non-null  int64  
 3   gender        200452 non-null  object 
 4   insurance     200452 non-null  object 
 5   grouped_race  200452 non-null  object 
 6   age_decile    200452 non-null  object 
 7   No Finding    200452 non-null  float64
 8   anchor_age    200452 non-null  int64  
dtypes: float64(1), int64(3), object(5)
memory usage: 13.8+ MB


In [74]:
df["anchor_age"].value_counts()

anchor_age
91    5889
66    5076
65    4859
63    4830
64    4792
      ... 
31     859
24     784
22     779
20     702
21     698
Name: count, Length: 71, dtype: int64

In [75]:
#how many samples correspond to female patients
print(df["gender"].value_counts())
df["gender"].value_counts(normalize=True)*100

gender
M    107241
F     93211
Name: count, dtype: int64


gender
M    53.499591
F    46.500409
Name: proportion, dtype: float64

In [76]:
M_df = df[df["gender"] == 'M']
F_df = df[df["gender"] == 'F']


In [77]:
#how many unique patients do i have in each dataframe?
# Check the number of unique values in a specific column
print(df["subject_id"].nunique())
print(M_df["subject_id"].nunique())
print(F_df["subject_id"].nunique())


46187
21865
24322


In [78]:
def print_age_info(df,M_df,F_df):
    # Calculate the mean of the column
    print(f'{df["anchor_age"].mean()}  {df["anchor_age"].std()}')
    print(f'{M_df["anchor_age"].mean()}  {M_df["anchor_age"].std()}')
    print(f'{F_df["anchor_age"].mean()}  {F_df["anchor_age"].std()}')
print_age_info(df,M_df,F_df)

62.486814798555265  16.516280298888052
62.19588590184724  15.740662147509413
62.821533939127356  17.359895496387747


In [79]:

def print_group_info(name,df,M_df,F_df):
    print("all")
    print(df[name].value_counts())
    print(np.round(df[name].value_counts(normalize=True)*100,1))
    print("male")
    print(M_df[name].value_counts())
    print(np.round(M_df[name].value_counts(normalize=True)*100,1))
    print("female")
    print(F_df[name].value_counts())
    print(np.round(F_df[name].value_counts(normalize=True)*100,1))

In [80]:
#how many of each ethnic group
print_group_info("grouped_race",df,M_df,F_df)

all
grouped_race
White              140658
Black               33615
Other               15143
Hispanic/Latino     11036
Name: count, dtype: int64
grouped_race
White              70.2
Black              16.8
Other               7.6
Hispanic/Latino     5.5
Name: proportion, dtype: float64
male
grouped_race
White              79455
Black              13759
Other               8563
Hispanic/Latino     5464
Name: count, dtype: int64
grouped_race
White              74.1
Black              12.8
Other               8.0
Hispanic/Latino     5.1
Name: proportion, dtype: float64
female
grouped_race
White              61203
Black              19856
Other               6580
Hispanic/Latino     5572
Name: count, dtype: int64
grouped_race
White              65.7
Black              21.3
Other               7.1
Hispanic/Latino     6.0
Name: proportion, dtype: float64


In [81]:
#how many of each ethnic group
print_group_info("age_decile",df,M_df,F_df)

all
age_decile
60-80    86635
40-60    60238
80+      33485
20-40    20094
Name: count, dtype: int64
age_decile
60-80    43.2
40-60    30.1
80+      16.7
20-40    10.0
Name: proportion, dtype: float64
male
age_decile
60-80    48073
40-60    33774
80+      15634
20-40     9760
Name: count, dtype: int64
age_decile
60-80    44.8
40-60    31.5
80+      14.6
20-40     9.1
Name: proportion, dtype: float64
female
age_decile
60-80    38562
40-60    26464
80+      17851
20-40    10334
Name: count, dtype: int64
age_decile
60-80    41.4
40-60    28.4
80+      19.2
20-40    11.1
Name: proportion, dtype: float64


In [82]:
#how many of each ethnic group
print_group_info("insurance",df,M_df,F_df)

all
insurance
Medicare    113625
Private      52440
Medicaid     34387
Name: count, dtype: int64
insurance
Medicare    56.7
Private     26.2
Medicaid    17.2
Name: proportion, dtype: float64
male
insurance
Medicare    59785
Private     29979
Medicaid    17477
Name: count, dtype: int64
insurance
Medicare    55.7
Private     28.0
Medicaid    16.3
Name: proportion, dtype: float64
female
insurance
Medicare    53840
Private     22461
Medicaid    16910
Name: count, dtype: int64
insurance
Medicare    57.8
Private     24.1
Medicaid    18.1
Name: proportion, dtype: float64


In [83]:
#how many of each ethnic group
print_group_info("No Finding",df,M_df,F_df)

all
No Finding
0.0    138042
1.0     62410
Name: count, dtype: int64
No Finding
0.0    68.9
1.0    31.1
Name: proportion, dtype: float64
male
No Finding
0.0    75931
1.0    31310
Name: count, dtype: int64
No Finding
0.0    70.8
1.0    29.2
Name: proportion, dtype: float64
female
No Finding
0.0    62111
1.0    31100
Name: count, dtype: int64
No Finding
0.0    66.6
1.0    33.4
Name: proportion, dtype: float64


In [84]:
train_df=df[df["subject_id"].isin(train_ids)]
train_M_df=M_df[M_df["subject_id"].isin(train_ids)]
train_F_df=F_df[F_df["subject_id"].isin(train_ids)]
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 160967 entries, 0 to 200451
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0    160967 non-null  int64  
 1   path          160967 non-null  object 
 2   subject_id    160967 non-null  int64  
 3   gender        160967 non-null  object 
 4   insurance     160967 non-null  object 
 5   grouped_race  160967 non-null  object 
 6   age_decile    160967 non-null  object 
 7   No Finding    160967 non-null  float64
 8   anchor_age    160967 non-null  int64  
dtypes: float64(1), int64(3), object(5)
memory usage: 12.3+ MB


In [85]:
#how many samples correspond to female patients
print(train_df["gender"].value_counts())
train_df["gender"].value_counts(normalize=True)*100

gender
M    86721
F    74246
Name: count, dtype: int64


gender
M    53.875018
F    46.124982
Name: proportion, dtype: float64

In [86]:
print(train_df["subject_id"].nunique())
print(train_M_df["subject_id"].nunique())
print(train_F_df["subject_id"].nunique())

36949
17547
19402


In [87]:
print_age_info(train_df,train_M_df,train_F_df)

62.45115458448005  16.565853572464206
62.19515457616956  15.786704289794159
62.75016835923821  17.427225657750096


In [88]:
#how many of each ethnic group
print_group_info("grouped_race",train_df,train_M_df,train_F_df)

all
grouped_race
White              112917
Black               26872
Other               12303
Hispanic/Latino      8875
Name: count, dtype: int64
grouped_race
White              70.1
Black              16.7
Other               7.6
Hispanic/Latino     5.5
Name: proportion, dtype: float64
male
grouped_race
White              64134
Black              11238
Other               6959
Hispanic/Latino     4390
Name: count, dtype: int64
grouped_race
White              74.0
Black              13.0
Other               8.0
Hispanic/Latino     5.1
Name: proportion, dtype: float64
female
grouped_race
White              48783
Black              15634
Other               5344
Hispanic/Latino     4485
Name: count, dtype: int64
grouped_race
White              65.7
Black              21.1
Other               7.2
Hispanic/Latino     6.0
Name: proportion, dtype: float64


In [89]:
#how many of each ethnic group
print_group_info("age_decile",train_df,train_M_df,train_F_df)

all
age_decile
60-80    69343
40-60    48210
80+      27059
20-40    16355
Name: count, dtype: int64
age_decile
60-80    43.1
40-60    30.0
80+      16.8
20-40    10.2
Name: proportion, dtype: float64
male
age_decile
60-80    39007
40-60    27095
80+      12585
20-40     8034
Name: count, dtype: int64
age_decile
60-80    45.0
40-60    31.2
80+      14.5
20-40     9.3
Name: proportion, dtype: float64
female
age_decile
60-80    30336
40-60    21115
80+      14474
20-40     8321
Name: count, dtype: int64
age_decile
60-80    40.9
40-60    28.4
80+      19.5
20-40    11.2
Name: proportion, dtype: float64


In [90]:
#how many of each ethnic group
print_group_info("insurance",train_df,train_M_df,train_F_df)

all
insurance
Medicare    90976
Private     41990
Medicaid    28001
Name: count, dtype: int64
insurance
Medicare    56.5
Private     26.1
Medicaid    17.4
Name: proportion, dtype: float64
male
insurance
Medicare    48318
Private     24279
Medicaid    14124
Name: count, dtype: int64
insurance
Medicare    55.7
Private     28.0
Medicaid    16.3
Name: proportion, dtype: float64
female
insurance
Medicare    42658
Private     17711
Medicaid    13877
Name: count, dtype: int64
insurance
Medicare    57.5
Private     23.9
Medicaid    18.7
Name: proportion, dtype: float64


In [91]:
#how many of each ethnic group
print_group_info("No Finding",train_df,train_M_df,train_F_df)

all
No Finding
0.0    110985
1.0     49982
Name: count, dtype: int64
No Finding
0.0    68.9
1.0    31.1
Name: proportion, dtype: float64
male
No Finding
0.0    61449
1.0    25272
Name: count, dtype: int64
No Finding
0.0    70.9
1.0    29.1
Name: proportion, dtype: float64
female
No Finding
0.0    49536
1.0    24710
Name: count, dtype: int64
No Finding
0.0    66.7
1.0    33.3
Name: proportion, dtype: float64


In [92]:
test_df=df[df["subject_id"].isin(test_ids)]
test_M_df=M_df[M_df["subject_id"].isin(test_ids)]
test_F_df=F_df[F_df["subject_id"].isin(test_ids)]
test_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39485 entries, 5 to 200431
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    39485 non-null  int64  
 1   path          39485 non-null  object 
 2   subject_id    39485 non-null  int64  
 3   gender        39485 non-null  object 
 4   insurance     39485 non-null  object 
 5   grouped_race  39485 non-null  object 
 6   age_decile    39485 non-null  object 
 7   No Finding    39485 non-null  float64
 8   anchor_age    39485 non-null  int64  
dtypes: float64(1), int64(3), object(5)
memory usage: 3.0+ MB


In [93]:
#how many samples correspond to female patients
print(test_df["gender"].value_counts())
test_df["gender"].value_counts(normalize=True)*100

gender
M    20520
F    18965
Name: count, dtype: int64


gender
M    51.969102
F    48.030898
Name: proportion, dtype: float64

In [94]:
print(test_df["subject_id"].nunique())
print(test_M_df["subject_id"].nunique())
print(test_F_df["subject_id"].nunique())

9238
4318
4920


In [95]:
print_age_info(test_df,test_M_df,test_F_df)

62.63218943902748  16.312029112596953
62.19897660818714  15.544954968020464
63.100922752438706  17.09134119931327


In [96]:
#how many of each ethnic group
print_group_info("grouped_race",test_df,test_M_df,test_F_df)

all
grouped_race
White              27741
Black               6743
Other               2840
Hispanic/Latino     2161
Name: count, dtype: int64
grouped_race
White              70.3
Black              17.1
Other               7.2
Hispanic/Latino     5.5
Name: proportion, dtype: float64
male
grouped_race
White              15321
Black               2521
Other               1604
Hispanic/Latino     1074
Name: count, dtype: int64
grouped_race
White              74.7
Black              12.3
Other               7.8
Hispanic/Latino     5.2
Name: proportion, dtype: float64
female
grouped_race
White              12420
Black               4222
Other               1236
Hispanic/Latino     1087
Name: count, dtype: int64
grouped_race
White              65.5
Black              22.3
Other               6.5
Hispanic/Latino     5.7
Name: proportion, dtype: float64


In [97]:
#how many of each ethnic group
print_group_info("age_decile",test_df,test_M_df,test_F_df)

all
age_decile
60-80    17292
40-60    12028
80+       6426
20-40     3739
Name: count, dtype: int64
age_decile
60-80    43.8
40-60    30.5
80+      16.3
20-40     9.5
Name: proportion, dtype: float64
male
age_decile
60-80    9066
40-60    6679
80+      3049
20-40    1726
Name: count, dtype: int64
age_decile
60-80    44.2
40-60    32.5
80+      14.9
20-40     8.4
Name: proportion, dtype: float64
female
age_decile
60-80    8226
40-60    5349
80+      3377
20-40    2013
Name: count, dtype: int64
age_decile
60-80    43.4
40-60    28.2
80+      17.8
20-40    10.6
Name: proportion, dtype: float64


In [98]:
#how many of each ethnic group
print_group_info("insurance",test_df,test_M_df,test_F_df)

all
insurance
Medicare    22649
Private     10450
Medicaid     6386
Name: count, dtype: int64
insurance
Medicare    57.4
Private     26.5
Medicaid    16.2
Name: proportion, dtype: float64
male
insurance
Medicare    11467
Private      5700
Medicaid     3353
Name: count, dtype: int64
insurance
Medicare    55.9
Private     27.8
Medicaid    16.3
Name: proportion, dtype: float64
female
insurance
Medicare    11182
Private      4750
Medicaid     3033
Name: count, dtype: int64
insurance
Medicare    59.0
Private     25.0
Medicaid    16.0
Name: proportion, dtype: float64


In [99]:
#how many of each ethnic group
print_group_info("No Finding",test_df,test_M_df,test_F_df)

all
No Finding
0.0    27057
1.0    12428
Name: count, dtype: int64
No Finding
0.0    68.5
1.0    31.5
Name: proportion, dtype: float64
male
No Finding
0.0    14482
1.0     6038
Name: count, dtype: int64
No Finding
0.0    70.6
1.0    29.4
Name: proportion, dtype: float64
female
No Finding
0.0    12575
1.0     6390
Name: count, dtype: int64
No Finding
0.0    66.3
1.0    33.7
Name: proportion, dtype: float64
